In [0]:
# if using ADLS

"""
spark.conf.set(
  "fs.azure.account.key.ecomprojectdatastorage.dfs.core.windows.net",
  "Storage Account Key"
)

dbutils.fs.ls("abfss://raw@ecomprojectdatastorage.dfs.core.windows.net/")
"""
 

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

# -------------------------------
#  Define Schemas
# -------------------------------

# Customers
customers_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("signup_date", StringType(), True),
    StructField("country", StringType(), True)
])

# Orders
orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("total_amount", DoubleType(), True)
])

# Products
products_schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_quantity", IntegerType(), True)
])


In [0]:
# -------------------------------
#  Read Data using Autoloader
# -------------------------------
customers_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    #.option("cloudFiles.inferSchema", "true")
    .option("header", "true")
    .schema(customers_schema)
    .load("/Volumes/ecom/storage/project/raw/customers/")
)

products_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    #.option("cloudFiles.inferSchema", "true")
    .schema(products_schema)
    .load("/Volumes/ecom/storage/project/raw/products/")
)

orders_df = (
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    #.option("cloudFiles.inferSchema", "true")
    .schema(orders_schema)
    .load("/Volumes/ecom/storage/project/raw/orders/")
)

In [0]:
# -------------------------------
#  Write to Bronze Delta Tables
# -------------------------------

customers_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/customers/") \
    .trigger(once=True) \
    .start("/Volumes/ecom/storage/project/bronze/customers/")

products_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/products/") \
    .trigger(once=True) \
    .start("/Volumes/ecom/storage/project/bronze/products/")

orders_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/ecom/storage/project/checkpoints/orders/") \
    .trigger(once=True) \
    .start("/Volumes/ecom/storage/project/bronze/orders/")

In [0]:
# viewing data

spark.read.format("delta") \
    .load("/Volumes/ecom/storage/project/bronze/customers/") \
    .show()
